# 02 — Parse Documents

## Purpose
Extracts text content from every pipeline-ready document using
`AI_PARSE_DOCUMENT` and populates two downstream tables:
`DOCUMENTS_TEXT` (one row per file, raw JSON value from Cortex) and
`DOCUMENTS_PAGES` (one row per page, plain text content ready for
translation and classification).

## What this notebook does
Queries `DOCUMENTS_INGESTED` for files with `STATUS = 'PENDING'` that
have not yet been parsed, runs `AI_PARSE_DOCUMENT` in `LAYOUT` mode on
each file in parallel (up to 4 concurrent Cortex calls), then flattens
the per-page content into `DOCUMENTS_PAGES`.

Image formats (`.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`) are parsed
without `page_split` since Cortex does not support it for those formats —
they always produce a single page row. All other formats use
`page_split: TRUE` to produce one row per page.

Only the `value` portion of the Cortex JSON response is stored —
`error` and `metadata` are discarded. Page population reads from
`DOCUMENTS_TEXT` directly so no Cortex calls are repeated if re-run.

Status is updated in `DOCUMENTS_INGESTED` to `PARSED` on success or
`PARSE_ERROR` on failure. 

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_TEXT` | One row per file — raw value JSON, page count |
| `PROCESSING.DOCUMENTS_PAGES` | One row per page — page number, text content |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `PARSED` or `PARSE_ERROR` |

In [ ]:
import io
import os
import json
import pytz
import pandas as pd
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session

RAW_STAGE         = '@PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE'
PDF_STAGE         = '@PERMAFROST_POC.PROCESSING.PROCESSED_DOC_STAGE'
DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'

PARSEABLE_FORMATS = {'.pdf', '.docx', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}
IMAGE_FORMATS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

s = get_active_session() 

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def now_ast(): # could be replace with any time zone later
    tz = pytz.timezone('America/Halifax')
    return datetime.now(tz).isoformat()

In [ ]:
# Parse documents 

# Non-image formats (PDF, DOCX) — page_split: TRUE 
pdf_results = s.sql(f"""
    SELECT
        i.DOC_ID,
        i.ORIGINAL_FILENAME,
        i.SOURCE_FORMAT,
        i.STAGE_PATH,
        AI_PARSE_DOCUMENT(
            TO_FILE(i.STAGE_PATH),
            {{'mode': 'LAYOUT', 'page_split': TRUE}},
            TRUE
        ) AS PARSED
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_TEXT t
        ON i.DOC_ID = t.DOC_ID
    WHERE i.STATUS        = 'PENDING'
      AND i.SOURCE_FORMAT IN ('.pdf', '.docx')
      AND t.DOC_ID        IS NULL
""").collect()

info(f"PDF/DOCX parse results: {len(pdf_results)} file(s)")

# Image formats — no page_split 
img_results = s.sql(f"""
    SELECT
        i.DOC_ID,
        i.ORIGINAL_FILENAME,
        i.SOURCE_FORMAT,
        i.STAGE_PATH,
        AI_PARSE_DOCUMENT(
            TO_FILE(i.STAGE_PATH),
            {{'mode': 'LAYOUT'}},
            TRUE
        ) AS PARSED
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_TEXT t
        ON i.DOC_ID = t.DOC_ID
    WHERE i.STATUS        = 'PENDING'
      AND i.SOURCE_FORMAT IN ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
      AND t.DOC_ID        IS NULL
""").collect()

info(f"Image parse results: {len(img_results)} file(s)")

all_results = pdf_results + img_results
info(f"Total parsed: {len(all_results)} file(s)")

if not all_results:
    print("\nNothing to parse.")


In [ ]:

# Process parse results
# Build DOCUMENTS_TEXT rows and DOCUMENTS_PAGES rows from raw JSON

text_rows    = []
page_rows    = []
llm_rows     = []
parse_errors = []
success_ids  = []
error_ids    = []

for row in all_results:
    doc_id   = row['DOC_ID']
    filename = row['ORIGINAL_FILENAME']
    raw_json = row['PARSED']

    try:
        if raw_json is None:
            raise ValueError("AI_PARSE_DOCUMENT returned NULL")

        result = json.loads(raw_json) if isinstance(raw_json, str) else raw_json

        if result.get('error'):
            raise ValueError(result['error'])

        value = result.get('value', {})
        pages = value.get('pages') or [
            {'index': 0, 'content': value.get('content', '')}
        ]

        sorted_pages = sorted(pages, key=lambda x: x['index'])

        # Full text for DOCUMENTS_TEXT
        full_text = '\n\n'.join(
            f"[PAGE {p['index'] + 1}]\n{p.get('content', '').strip()}"
            for p in sorted_pages
            if p.get('content', '').strip()
        )

        non_empty_pages = [
            p for p in sorted_pages
            if p.get('content', '').strip()
        ]

        page_count = len(non_empty_pages)

        # DOCUMENTS_TEXT row
        text_rows.append({
            'DOC_ID':              doc_id,
            'EXTRACTED_TEXT':      full_text,
            'RAW_EXTRACTED_VALUE': json.dumps(value),
            'PAGE_COUNT':          page_count,
            'EXTRACTION_MODEL':    'AI_PARSE_DOCUMENT',
            'EXTRACTED_AT':        now_ast(),
        })

        # DOCUMENTS_PAGES rows — one per non-empty page
        for p in non_empty_pages:
            page_rows.append({
                'DOC_ID':                  doc_id,
                'PAGE_INDEX':              p['index'],
                'PAGE_NUMBER':             p['index'] + 1,
                'PAGE_CONTENT':            p.get('content', '').strip(),
                'PAGE_CONTENT_TRANSLATED': None,
                'PAGE_CONTENT_REDACTED':   None,
            })

        success_ids.append(doc_id)
        info(f"  [OK]   {filename} — {page_count} page(s), "
             f"{len(full_text)} chars")

    except Exception as e:
        error_ids.append(doc_id)
        parse_errors.append({'doc_id': doc_id, 'file': filename, 'error': str(e)})
        error(f"  [FAIL] {filename}: {e}")


# Write DOCUMENTS_TEXT

if text_rows:
    s.write_pandas(
        pd.DataFrame(text_rows),
        table_name='DOCUMENTS_TEXT',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(text_rows)} row(s) to DOCUMENTS_TEXT")

# Write DOCUMENTS_PAGES
if page_rows:
    s.write_pandas(
        pd.DataFrame(page_rows),
        table_name='DOCUMENTS_PAGES',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(page_rows)} row(s) to DOCUMENTS_PAGES")


# Update STATUS in DOCUMENTS_INGESTED

if success_ids:
    id_list = ','.join(f"'{i}'" for i in success_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'PARSED'
        WHERE DOC_ID IN ({id_list})
    """).collect()

if error_ids:
    id_list = ','.join(f"'{i}'" for i in error_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'PARSE_ERROR'
        WHERE DOC_ID IN ({id_list})
    """).collect()


# Summary

print(f"\n Parse summary")
print(f"  Parsed successfully : {len(text_rows)}")
print(f"  Pages written       : {len(page_rows)}")
print(f"  Errors              : {len(parse_errors)}")
print(f"  Total input tokens  : {sum(r['TOKENS_IN'] for r in llm_rows):,}")

if parse_errors:
    print("\n  Failed files:")
    for e in parse_errors:
        print(f"    {e['file']}: {e['error']}")

print(f"\n DOCUMENTS_TEXT coverage")
s.sql(f"""
    SELECT
        i.SOURCE_FORMAT,
        COUNT(*)          AS TOTAL_FILES,
        COUNT(t.DOC_ID)   AS PARSED_FILES,
        AVG(t.PAGE_COUNT) AS AVG_PAGES
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_TEXT t
        ON i.DOC_ID = t.DOC_ID
    WHERE i.SOURCE_FORMAT IN (
        '.pdf', '.docx', '.jpg', '.jpeg', '.png', '.tif', '.tiff'
    )
    GROUP BY i.SOURCE_FORMAT
    ORDER BY TOTAL_FILES DESC
""").show()